In [2]:
%cd "D:\Coding\projects\BatteryLifePrediction"

D:\Coding\projects\BatteryLifePrediction


In [ ]:
from torch.utils.data import DataLoader

from src.data import TongjiTaskDataset, collate_tongji

dataset = TongjiTaskDataset.from_config("configs/basic.yaml")

In [ ]:
task_position = dataset.tasks.index(("Tongji1_CY25-025_1--4", 50))
sample_position = task_position * dataset.samples_per_task
sample = dataset[sample_position, 1]
print(sample["last_cycle_position"], sample["last_cycle_id"])

loader = DataLoader(
    dataset,
    batch_size=2,
    sampler=[(0, 1), (sample_position, 1)],
    collate_fn=collate_tongji,
    num_workers=0,
)

In [ ]:
batch = next(iter(loader))

print(batch["values"].shape)  # (B, N_max, T_max, F)
print(batch["point_valid"].shape)  # (B, N_max, T_max)
print(batch["cycle_valid"])  # (B, N_max)
print(batch["target_soh"].shape)  # (B, forecast_horizon_cycles)

In [5]:
from itertools import cycle

from src.data.loading import build_datasets, make_loader
from src.utils.files import read_yaml

config = read_yaml("configs/basic.yaml")

# 可选：在 Notebook 中增加每个任务的样本数和批大小，方便观察同一任务在一批中重复出现。
config["data"]["samples_per_task"] = 10
config["data"]["batch_size"] = 32

datasets, _ = build_datasets(config)
train_dataset = datasets["train"]
train_loader = make_loader(train_dataset, config, shuffle=True)


def preview_batches(loader, max_epochs):
    for epoch in cycle(range(max_epochs)):
        loader.sampler.set_epoch(epoch)
        for batch_position, batch in enumerate(loader, start=1):
            yield epoch, batch_position, batch


batch_stream = preview_batches(train_loader, train_dataset.max_epochs)

In [12]:
from statistics import mean

from IPython.display import Markdown, display

epoch, batch_position, batch = next(batch_stream)

cycle_counts = batch["cycle_valid"].sum(dim=-1).tolist()
segment_lengths = batch["point_valid"].sum(dim=-1)
valid_target_counts = batch["target_soh_valid"].sum(dim=-1).tolist()

tasks = {}
all_segment_lengths = []

for sample_position, (cell_id, last_cycle_position, last_cycle_id) in enumerate(
    zip(
        batch["cell_id"],
        batch["last_cycle_position"].tolist(),
        batch["last_cycle_id"].tolist(),
    )
):
    task = tasks.setdefault(
        (cell_id, last_cycle_position),
        {"last_cycle_id": last_cycle_id, "cycles": [], "points": [], "targets": []},
    )
    lengths = segment_lengths[sample_position][
        batch["cycle_valid"][sample_position]
    ].tolist()

    task["cycles"].append(cycle_counts[sample_position])
    task["points"].extend(lengths)
    task["targets"].append(valid_target_counts[sample_position])
    all_segment_lengths.extend(lengths)


def min_max_mean(values):
    return f"{min(values)} / {max(values)} / {mean(values):.2f}"


rows = [
    "| 电芯 | 最后循环位置 | 真实循环号 | 样本数 | 每样本循环数 | 每片段点数 | 每样本有效 SOH 数 |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
]

for (cell_id, position), task in sorted(tasks.items()):
    rows.append(
        f"| {cell_id} | {position} | {task['last_cycle_id']} "
        f"| {len(task['cycles'])} | {min_max_mean(task['cycles'])} "
        f"| {min_max_mean(task['points'])} | {min_max_mean(task['targets'])} |"
    )

summary = (
    f"**Epoch {epoch} · 第 {batch_position} 批**　"
    f"{len(cycle_counts)} 份样本，覆盖 {len(tasks)} 个任务\n\n"
    f"- 每份样本平均包含 **{mean(cycle_counts):.2f}** 个充电循环\n"
    f"- 每个充电片段平均包含 **{mean(all_segment_lengths):.2f}** 个采样点\n\n"
    "表中的统计顺序为 **最小 / 最大 / 平均**。\n\n"
)

display(Markdown(summary + "\n".join(rows)))

**Epoch 0 · 第 7 批**　32 份样本，覆盖 22 个任务

- 每份样本平均包含 **3.09** 个充电循环
- 每个充电片段平均包含 **316.96** 个采样点

表中的统计顺序为 **最小 / 最大 / 平均**。

| 电芯 | 最后循环位置 | 真实循环号 | 样本数 | 每样本循环数 | 每片段点数 | 每样本有效 SOH 数 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Tongji1_CY25-025_1--4 | 1 | 2 | 1 | 1 / 1 / 1.00 | 321 / 321 / 321.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 3 | 4 | 2 | 1 / 1 / 1.00 | 321 / 321 / 321.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 5 | 6 | 1 | 1 / 1 / 1.00 | 320 / 320 / 320.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 6 | 7 | 1 | 1 / 1 / 1.00 | 321 / 321 / 321.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 9 | 10 | 1 | 1 / 1 / 1.00 | 320 / 320 / 320.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 11 | 12 | 1 | 2 / 2 / 2.00 | 319 / 320 / 319.50 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 14 | 15 | 1 | 2 / 2 / 2.00 | 318 / 321 / 319.50 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 18 | 19 | 2 | 2 / 2 / 2.00 | 317 / 319 / 318.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 21 | 22 | 1 | 3 / 3 / 3.00 | 316 / 321 / 318.67 | 3 / 3 / 3.00 |
| Tongji1_CY25-025_1--4 | 22 | 23 | 3 | 3 / 3 / 3.00 | 316 / 321 / 317.67 | 3 / 3 / 3.00 |
| Tongji1_CY25-025_1--4 | 27 | 29 | 1 | 3 / 3 / 3.00 | 316 / 321 / 319.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 28 | 30 | 1 | 3 / 3 / 3.00 | 316 / 321 / 318.33 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 29 | 31 | 2 | 3 / 3 / 3.00 | 314 / 321 / 316.83 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 30 | 32 | 1 | 3 / 3 / 3.00 | 315 / 318 / 316.33 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 32 | 34 | 3 | 4 / 4 / 4.00 | 314 / 321 / 316.17 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 35 | 37 | 1 | 4 / 4 / 4.00 | 314 / 322 / 318.00 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 39 | 41 | 1 | 4 / 4 / 4.00 | 313 / 318 / 315.75 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 40 | 42 | 2 | 4 / 4 / 4.00 | 313 / 321 / 315.25 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 41 | 43 | 1 | 5 / 5 / 5.00 | 313 / 318 / 315.60 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 42 | 44 | 1 | 5 / 5 / 5.00 | 313 / 318 / 315.20 | 4 / 4 / 4.00 |
| Tongji1_CY25-025_1--4 | 48 | 50 | 1 | 5 / 5 / 5.00 | 311 / 320 / 316.40 | 3 / 3 / 3.00 |
| Tongji1_CY25-025_1--4 | 50 | 53 | 3 | 5 / 5 / 5.00 | 312 / 321 / 316.07 | 4 / 4 / 4.00 |